# Stage 4 - Models: is there a border premium, and did it change?

**Input:** `data/processed/sales_features.csv` (from `03_features.ipynb`)
**Output:** four tables in `data/processed/`: `model_comparison.csv`, `boosting_predictions.csv`, `border_shape.csv`, `border_effect_by_year.csv`

Plan:

1. One target for all models: the price of a home relative to the homes sold in the same quarter.
2. One test for all models: predict municipalities the model has never seen.
3. Three models, from simple to flexible: linear regression, random forest, gradient boosting.
4. Each model is fitted four times: without any border variable, then with one of three ways to measure the border. If a border variable helps to predict unseen municipalities, the border carries information that the other variables do not.
5. For the best model: the shape of the border effect, and its size year by year.

*Tool declaration: this notebook was drafted with Claude (Anthropic), asked to write the model comparison, the tuning loops, and the yearly border-effect analysis described above.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold

from helpers import (CONTROL_FEATURES, N_FOLDS, PROCESSED, PROPERTY_FEATURES, SEED,
                     evaluate_by_commune, load_model_data)

sales = load_model_data()
y = sales["price_vs_quarter"]
communes = sales["code_commune"]
print(f"{len(sales):,} sales, {communes.nunique()} municipalities")

## 1. Target and variables

**Target.** `price_vs_quarter` is the log price per m2 minus the average log price of the quarter of sale. Homes are therefore only compared with homes sold at the same time, which removes market-wide shocks.

**Variables.** Every model gets the same description of the home and of its municipality. The border variable changes from one specification to the next:

| Specification | Border variable |
|---|---|
| `no border` | none |
| `any crossing` | distance to the nearest crossing, any canton (the definition of the proposal) |
| `Geneva crossing` | distance to the nearest crossing into the canton of Geneva |
| `cross-border share` | share of the municipality's workers employed in Switzerland |

Latitude and longitude are left out on purpose: with them, a flexible model could rebuild the distance to the border by itself and the border variable would look useless. Location is the job of the Gaussian process in stage 5.

In [ ]:
base_features = PROPERTY_FEATURES + CONTROL_FEATURES

specifications = {
    "no border": base_features,
    "any crossing": base_features + ["dist_border_km"],
    "Geneva crossing": base_features + ["dist_geneva_km"],
    "cross-border share": base_features + ["cross_border_share"],
}

results = []  # one row per (model, specification)

def run_all_specifications(model, model_name):
    """Evaluate one model on the four specifications and store the scores."""
    for spec_name, features in specifications.items():
        scores, _ = evaluate_by_commune(model, sales[features], y, communes)
        results.append({"model": model_name, "specification": spec_name, **scores})
        print(f"{model_name:18s} {spec_name:20s} R2 = {scores['R2']:.3f}   RMSE = {scores['RMSE']:.4f}")

## 2. How the models are tested

The question is about places, so the test must be about places: a model that has seen other sales from the same municipality can simply memorise its price level. The municipalities are split in 5 groups; the model is trained on 4 groups and predicts the 5th, in turn (`evaluate_by_commune` in `helpers.py`). Every score below is therefore measured on municipalities the model never saw.

Scores: R2 (share of the price differences explained), RMSE and MAE (typical error, in log points: 0.10 is about 10%).

## 3. Linear regression

The simplest model: each variable moves the price by a fixed percentage.

In [ ]:
run_all_specifications(LinearRegression(), "Linear regression")

Coefficients of the `Geneva crossing` specification, fitted on all sales. A coefficient of -0.01 on a distance means: one more kilometre, 1% cheaper.

In [ ]:
features = specifications["Geneva crossing"]
linear = LinearRegression().fit(sales[features], y)
pd.DataFrame({
    "coefficient": linear.coef_,
    "effect of +1 standard deviation": linear.coef_ * sales[features].std(),
}, index=features).round(4)

## 4. Random forest

A random forest averages many decision trees. It can represent what the linear model cannot: an effect that is strong in the first kilometres and then flat, or a border effect that differs between a village and a town.

One setting is tuned, the minimum number of sales per leaf: small leaves follow the training data closely, large leaves give smoother predictions. Three values are compared with the same test as above. Each tree is grown on half of the sales to keep the running time short.

In [ ]:
def random_forest(min_samples_leaf):
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=min_samples_leaf,
                                 max_features=0.5, max_samples=0.5, n_jobs=-1, random_state=SEED)

tuning_features = specifications["Geneva crossing"]
forest_tuning = {}
for leaf in [5, 20, 50]:
    scores, _ = evaluate_by_commune(random_forest(leaf), sales[tuning_features], y, communes)
    forest_tuning[leaf] = scores["RMSE"]
    print(f"min_samples_leaf = {leaf:3d}   RMSE = {scores['RMSE']:.4f}")

best_leaf = min(forest_tuning, key=forest_tuning.get)
print("Chosen:", best_leaf)

In [ ]:
run_all_specifications(random_forest(best_leaf), "Random forest")

## 5. Gradient boosting

Gradient boosting also uses trees, but builds them one after the other, each one correcting the errors of the previous ones. The setting tuned here is the size of each tree (number of leaves): small trees learn simple rules, large trees learn detailed interactions and can overfit the municipalities of the training set.

In [ ]:
def gradient_boosting(max_leaf_nodes):
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                         max_leaf_nodes=max_leaf_nodes, random_state=SEED)

boosting_tuning = {}
for leaves in [7, 15, 31]:
    scores, _ = evaluate_by_commune(gradient_boosting(leaves), sales[tuning_features], y, communes)
    boosting_tuning[leaves] = scores["RMSE"]
    print(f"max_leaf_nodes = {leaves:3d}   RMSE = {scores['RMSE']:.4f}")

best_leaves = min(boosting_tuning, key=boosting_tuning.get)
print("Chosen:", best_leaves)

In [ ]:
run_all_specifications(gradient_boosting(best_leaves), "Gradient boosting")

## 6. Comparison

R2 on unseen municipalities, for each model and each way to measure the border. Two things to read:

- across a row: does adding a border variable improve the prediction compared with `no border`?
- down a column: does a more flexible model do better than the linear regression?

The settings of the forest and of the boosting were chosen with the same test that gives these scores, so their scores are slightly optimistic. Only three values were tried for each, which keeps this effect small.

In [ ]:
comparison = pd.DataFrame(results)
comparison.to_csv(PROCESSED / "model_comparison.csv", index=False)

order = list(specifications)
comparison.pivot(index="model", columns="specification", values="R2")[order].round(3)

In [ ]:
comparison.pivot(index="model", columns="specification", values="RMSE")[order].round(4)

The out-of-sample predictions of the gradient boosting, without border variable and with the distance to Geneva, are saved for stage 5.

In [ ]:
predictions = pd.DataFrame({"id_mutation": sales["id_mutation"]})
for column, spec_name in [("prediction_no_border", "no border"), ("prediction_geneva", "Geneva crossing")]:
    _, predictions[column] = evaluate_by_commune(
        gradient_boosting(best_leaves), sales[specifications[spec_name]], y, communes)
predictions.to_csv(PROCESSED / "boosting_predictions.csv", index=False)

## 7. Shape of the border effect

For the gradient boosting with the distance to Geneva: what price does the model predict when the distance is changed and everything else is kept as it is? This is a partial dependence curve. The curve is centred on its value at 20 km, so it reads directly as "premium compared with a similar home 20 km from the border".

Caution: the curve also moves homes to distances where such homes do not exist (a ski-resort flat placed 2 km from Geneva). To limit this, it is computed on sales within 30 km of a Geneva crossing only.

In [ ]:
MAX_DISTANCE = 30   # km: the partial dependence is computed on sales closer than this
NEAR, FAR = 2, 20   # km: the premium is the predicted price at NEAR minus the one at FAR

features = specifications["Geneva crossing"]
near_geneva = sales["dist_geneva_km"] <= MAX_DISTANCE
grid = np.arange(0.5, MAX_DISTANCE + 0.5, 0.5)

def distance_curve(model, X):
    """Average prediction of the model when dist_geneva_km is set to each value of the grid
    for every sale of X, all other variables unchanged (partial dependence)."""
    X = X.copy()
    averages = []
    for distance in grid:
        X["dist_geneva_km"] = distance
        averages.append(model.predict(X).mean())
    return np.array(averages)

boosting = gradient_boosting(best_leaves).fit(sales[features], y)
curve = distance_curve(boosting, sales.loc[near_geneva, features])
curve = curve - curve[grid == FAR][0]

shape = pd.DataFrame({"dist_geneva_km": grid, "premium_vs_20km": curve})
shape.to_csv(PROCESSED / "border_shape.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(grid, 100 * (np.exp(curve) - 1))
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Distance to nearest Geneva crossing (km)")
ax.set_ylabel("Predicted premium vs. 20 km (%)")
ax.set_title("Border premium by distance (gradient boosting, all years)")
plt.show()

## 8. Did the border effect change between 2021 and 2025?

The same analysis is repeated on each year separately, with three measures:

- **Linear coefficient** on the distance to Geneva (per km).
- **Permutation importance** of the distance to Geneva in the gradient boosting: the distances are shuffled between the sales of the unseen municipalities, and the increase of the error (RMSE) is recorded. If the model does not need the border to predict prices, shuffling changes nothing.
- **Premium at 2 km vs. 20 km** predicted by the gradient boosting, in %.

The last two are computed in each of the 5 groups of municipalities, which gives 5 values per year: the table reports their mean and their standard deviation, a rough measure of how sure the estimate is.

In [ ]:
rows = []
for year in sorted(sales["year"].unique()):
    this_year = sales[sales["year"] == year]
    X_year, y_year, communes_year = this_year[features], y[this_year.index], this_year["code_commune"]

    linear_year = LinearRegression().fit(X_year, y_year)
    linear_coefficient = linear_year.coef_[features.index("dist_geneva_km")]

    importances, premiums = [], []
    for train, test in GroupKFold(N_FOLDS).split(X_year, groups=communes_year):
        model = gradient_boosting(best_leaves).fit(X_year.iloc[train], y_year.iloc[train])

        shuffled = permutation_importance(model, X_year.iloc[test], y_year.iloc[test],
                                          scoring="neg_root_mean_squared_error",
                                          n_repeats=5, random_state=SEED)
        importances.append(shuffled.importances_mean[features.index("dist_geneva_km")])

        test_near = X_year.iloc[test]
        test_near = test_near[test_near["dist_geneva_km"] <= MAX_DISTANCE]
        year_curve = distance_curve(model, test_near)
        premiums.append(year_curve[grid == NEAR][0] - year_curve[grid == FAR][0])

    rows.append({
        "year": year,
        "sales": len(this_year),
        "linear_coefficient_per_km": linear_coefficient,
        "permutation_importance_mean": np.mean(importances),
        "permutation_importance_std": np.std(importances),
        "premium_2km_vs_20km_pct_mean": 100 * (np.exp(np.mean(premiums)) - 1),
        "premium_2km_vs_20km_pct_std": 100 * np.std(premiums),
    })

by_year = pd.DataFrame(rows)
by_year.to_csv(PROCESSED / "border_effect_by_year.csv", index=False)
by_year.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].errorbar(by_year["year"], by_year["permutation_importance_mean"],
                 yerr=by_year["permutation_importance_std"], marker="o", capsize=4)
axes[0].axhline(0, color="grey", linewidth=0.8)
axes[0].set_xlabel("Year of sale")
axes[0].set_ylabel("Increase in RMSE when distance is shuffled")
axes[0].set_title("Permutation importance of the distance to Geneva")

axes[1].errorbar(by_year["year"], by_year["premium_2km_vs_20km_pct_mean"],
                 yerr=by_year["premium_2km_vs_20km_pct_std"], marker="o", capsize=4)
axes[1].axhline(0, color="grey", linewidth=0.8)
axes[1].set_xlabel("Year of sale")
axes[1].set_ylabel("Premium at 2 km vs. 20 km (%)")
axes[1].set_title("Predicted border premium")

for ax in axes:
    ax.set_xticks(by_year["year"])
plt.tight_layout()
plt.show()